# ⚖️ debim: 407-Model Balanced 3D Visual Regression Benchmark

[![GitHub Repository](https://img.shields.io/badge/GitHub-PRIDA--TAKON%2Fdebim-181717?logo=github&style=for-the-badge)](https://github.com/PRIDA-TAKON/debim)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg?style=for-the-badge)](https://github.com/PRIDA-TAKON/debim/blob/main/LICENSE)

> **debim** Balanced Visual Benchmark Engine.
> Solves **Class Imbalance Bias** and **High-Frequency Redundancy** (e.g. thousands of identical pipe elbows/tees) via **Geometry Variant Deduplication** and **Balanced Macro-Averaging**.
>
> Every unique element variant is rigorously tested (1-2 samples per variant), ensuring unbiased fidelity evaluation across Structural, Architectural, and MEP systems.

In [ ]:
# 1. Install dependencies & clone latest debim repository
!pip install -q ifcopenshell "pydantic>=2.0" pyyaml rich trimesh matplotlib pillow pandas tabulate seaborn

!rm -rf /kaggle/working/debim
!git clone https://github.com/PRIDA-TAKON/debim.git /kaggle/working/debim

import sys
sys.path.insert(0, "/kaggle/working/debim/src")
sys.path.insert(0, "/kaggle/working/debim")
print("Environment initialized successfully with latest debim commit.")

In [ ]:
# 2. Discover and index all unique real-world IFC models
from pathlib import Path
import pandas as pd

input_root = Path("/kaggle/input")
all_ifc_files = sorted(list(input_root.glob("**/*.ifc")))
print(f"Discovered {len(all_ifc_files)} raw IFC files in input datasets.")

# Deduplicate by (filename, file_size)
unique_models = []
seen_signatures = set()
for f in all_ifc_files:
    size = f.stat().st_size
    if size < 5000: # Skip empty or micro snippets (<5KB)
        continue
    sig = (f.name, size)
    if sig not in seen_signatures:
        seen_signatures.add(sig)
        unique_models.append(f)

print(f"Indexed {len(unique_models)} unique benchmark models for balanced 3D visual testing.")

In [ ]:
# 3. Parallel 4-Core Balanced 3D Visual Regression Engine
import concurrent.futures
import os
import time
from pathlib import Path
from tools.render_visual_regression import run_visual_regression

output_root = Path("/kaggle/working/visual_regression_output")
output_root.mkdir(parents=True, exist_ok=True)

def worker_task(ifc_path_str):
    ifc_p = Path(ifc_path_str)
    model_dir = output_root / ifc_p.stem
    try:
        stat = run_visual_regression(
            ifc_path=ifc_p,
            output_dir=model_dir,
            max_elements_per_type=20,
            limit_total=50,
            save_images=True,
            save_passes_sample_limit=1,
            dedup_by_variant=True,
            max_samples_per_variant=2,
        )
        return {"model_name": ifc_p.name, "size_kb": round(ifc_p.stat().st_size / 1024, 1), **stat}
    except Exception as e:
        return {
            "model_name": ifc_p.name,
            "size_kb": round(ifc_p.stat().st_size / 1024, 1),
            "total_tested": 0,
            "pass_count": 0,
            "avg_match": 0.0,
            "macro_avg_match": 0.0,
            "structural_match": None,
            "mep_match": None,
            "avg_iou": 0.0,
            "macro_avg_iou": 0.0,
            "class_summary": {},
            "report_path": None,
            "duration": 0.0,
            "status": "FATAL_ERROR",
            "error": str(e)
        }

print(f"Launching 4-worker parallel execution across {len(unique_models)} models...")
t_start = time.time()
results = []
completed_count = 0

with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
    future_to_model = {executor.submit(worker_task, str(m)): m for m in unique_models}
    for future in concurrent.futures.as_completed(future_to_model):
        completed_count += 1
        res = future.result()
        results.append(res)
        if completed_count % 20 == 0 or completed_count == len(unique_models):
            elapsed = time.time() - t_start
            pct = completed_count / len(unique_models) * 100
            eta = (elapsed / completed_count) * (len(unique_models) - completed_count) / 60
            model_short = res['model_name'][:28]
            macro_val = res.get('macro_avg_match', 0)
            tested_val = res.get('total_tested', 0)
            print(f"[{completed_count:3d}/{len(unique_models)}] {pct:5.1f}% | Elapsed: {elapsed/60:4.1f}m | ETA: {eta:4.1f}m | Last: {model_short} (Macro: {macro_val:.1f}%, Tested: {tested_val})")

print(f"All {len(unique_models)} models processed in {(time.time() - t_start)/60:.2f} minutes.")

In [ ]:
# 4. Generate Master Balanced Analytics & Visual KPI Report
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

df = pd.DataFrame(results)
leaderboard_path = Path("/kaggle/working/visual_regression_balanced_leaderboard.csv")
df.to_csv(leaderboard_path, index=False)
print(f"Saved master leaderboard to: {leaderboard_path}")

tested_df = df[df["total_tested"] > 0]
total_pieces = int(tested_df["total_tested"].sum()) if len(tested_df) > 0 else 0
total_passed = int(tested_df["pass_count"].sum()) if len(tested_df) > 0 else 0
global_pass_rate = (total_passed / total_pieces * 100) if total_pieces > 0 else 0.0

macro_mean = float(tested_df["macro_avg_match"].mean()) if len(tested_df) > 0 else 0.0
micro_mean = float(tested_df["avg_match"].mean()) if len(tested_df) > 0 else 0.0
macro_iou = float(tested_df["macro_avg_iou"].mean()) if len(tested_df) > 0 else 0.0

struct_df = tested_df[tested_df["structural_match"].notnull()]
struct_mean = float(struct_df["structural_match"].mean()) if len(struct_df) > 0 else 0.0

mep_df = tested_df[tested_df["mep_match"].notnull()]
mep_mean = float(mep_df["mep_match"].mean()) if len(mep_df) > 0 else 0.0

display(Markdown(f"""
# 🏆 debim Balanced 3D Visual Regression Benchmark Results

| Metric | Score | Note |
|---|---|---|
| **Total Building Models Evaluated** | `{len(df)}` | 100% of benchmark set |
| **Models with Renderable Elements** | `{len(tested_df)}` | Physical BIM geometry detected |
| **Total Unique Variant Samples Tested** | `{total_pieces:,}` | Deduplicated across types |
| **Total Components Passing (>=85%)** | `{total_passed:,}` | Pass Rate: **{global_pass_rate:.1f}%** |
| **⚖️ Macro Average Visual Match (Unweighted Class Mean)** | **{macro_mean:.1f}%** | **Zero Class Imbalance Bias** |
| **📊 Micro Average Visual Match (Raw Element Mean)** | **{micro_mean:.1f}%** | All individual pieces |
| **🏗️ Structural Accuracy (Columns, Beams, Walls, Slabs)** | **{struct_mean:.1f}%** | Core load-bearing elements |
| **⚡ MEP Accuracy (Pipes, Ducts, Fittings)** | **{mep_mean:.1f}%** | Distribution & fitting systems |
| **📦 Mean 3D Bounding Box IoU** | **{macro_iou:.1f}%** | Volumetric placement alignment |
"""))

# Aggregate Category Breakdown across all models
cat_stats = {}
for r in results:
    summary = r.get("class_summary", {})
    for cls, cdata in summary.items():
        cat_stats.setdefault(cls, {"count": 0, "matches": [], "ious": []})
        cat_stats[cls]["count"] += cdata["count"]
        cat_stats[cls]["matches"].append(cdata["avg_match"])
        cat_stats[cls]["ious"].append(cdata["avg_iou"])

cat_rows = []
for cls, cd in cat_stats.items():
    cat_rows.append({
        "Element Class": cls,
        "Samples Tested": cd["count"],
        "Mean Match (%)": round(float(np.mean(cd["matches"])), 1),
        "Mean IoU (%)": round(float(np.mean(cd["ious"])), 1),
    })

cat_df = pd.DataFrame(cat_rows).sort_values("Samples Tested", ascending=False)
print("\n--- Per-Class Accuracy Breakdown across 407 Models ---")
display(cat_df)
cat_df.to_csv("/kaggle/working/class_accuracy_breakdown.csv", index=False)

# Top Models
print("\n--- Top 15 Best Fidelity Models ---")
display(tested_df[["model_name", "total_tested", "pass_count", "macro_avg_match", "avg_match", "structural_match", "mep_match"]].sort_values("macro_avg_match", ascending=False).head(15))
